> **Catatan (diperbarui):** notebook ini adalah catatan eksplorasi dan analisis awal. Pipeline final yang dipakai aplikasi ada di folder `src/` dan dijalankan lewat `retrain.py`. Angka final (MAPE, rentang 80%, dan pilihan model vs harga bulan lalu per komoditas) ada di folder `results/`. Beberapa sel di bawah mungkin menggunakan kriteria seleksi lama (MAE) dan fitur lama (`geo_id`, `month`).

# 03 - Feature Engineering
Lag 1/2/3/12, rolling mean 3/6 (hanya data sebelum bulan target), per pasar x komoditas.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd, numpy as np
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

In [ ]:
from src.data_loader import load_long
from src.feature_engineering import build_features, FEATURES
long = load_long()
feat = build_features(long)
d = feat[feat.target.notna() & feat.eligible]
print(len(feat), len(d))
d.head()

In [ ]:
# Uji kebocoran: lag_1 harus sama dengan target bulan sebelumnya pada grup yang sama
g = feat[(feat.geo_id==feat.geo_id.iloc[5000]) & (feat.commodity==feat.commodity.iloc[5000])].reset_index(drop=True)
m = g.target.iloc[:-1].notna().values
print('lag_1 == target bulan lalu:', (g.price_lag_1.iloc[1:].values[m] == g.target.iloc[:-1].values[m]).all())

In [ ]:
d.groupby(d.date.dt.year).size()